In [2]:
import pandas as pd

In [9]:
df = pd.read_csv('data_football_ratings.csv')

In [10]:
# Mantener solo las filas donde is_human sea diferente de 1
df = df[df["is_human"] != 1]
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 23756 entries, 1 to 50651
Data columns (total 63 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   competition             23756 non-null  object 
 1   date                    23756 non-null  object 
 2   match                   23756 non-null  object 
 3   team                    23756 non-null  object 
 4   pos                     23756 non-null  object 
 5   pos_role                23756 non-null  object 
 6   player                  23756 non-null  object 
 7   rater                   23756 non-null  object 
 8   is_human                23756 non-null  int64  
 9   original_rating         23756 non-null  float64
 10  goals                   23756 non-null  int64  
 11  assists                 23756 non-null  int64  
 12  shots_ontarget          23756 non-null  int64  
 13  shots_offtarget         23756 non-null  int64  
 14  shotsblocked            23756 non-null  int

In [11]:
df = df[df["rater"]=='WhoScored']
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 21354 entries, 1 to 50651
Data columns (total 63 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   competition             21354 non-null  object 
 1   date                    21354 non-null  object 
 2   match                   21354 non-null  object 
 3   team                    21354 non-null  object 
 4   pos                     21354 non-null  object 
 5   pos_role                21354 non-null  object 
 6   player                  21354 non-null  object 
 7   rater                   21354 non-null  object 
 8   is_human                21354 non-null  int64  
 9   original_rating         21354 non-null  float64
 10  goals                   21354 non-null  int64  
 11  assists                 21354 non-null  int64  
 12  shots_ontarget          21354 non-null  int64  
 13  shots_offtarget         21354 non-null  int64  
 14  shotsblocked            21354 non-null  int

In [12]:
df_porteros = df[df['pos'] == 'GK'].copy()

In [13]:
cols_to_drop = [
    "competition", "date", "match", "team", "player", "rater", "is_human",
    "degree_centrality", "betweenness_centrality", "closeness_centrality",
    "flow_centrality", "flow_success", "betweenness2goals","pos_role", "pos"
]

In [17]:
df_porteros = df_porteros.drop(columns=cols_to_drop)
df_porteros.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1571 entries, 47 to 50613
Data columns (total 48 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   original_rating   1571 non-null   float64
 1   goals             1571 non-null   int64  
 2   assists           1571 non-null   int64  
 3   shots_ontarget    1571 non-null   int64  
 4   shots_offtarget   1571 non-null   int64  
 5   shotsblocked      1571 non-null   int64  
 6   chances2score     1571 non-null   int64  
 7   drib_success      1571 non-null   int64  
 8   drib_unsuccess    1571 non-null   int64  
 9   keypasses         1571 non-null   int64  
 10  touches           1571 non-null   int64  
 11  passes_acc        1571 non-null   int64  
 12  passes_inacc      1571 non-null   int64  
 13  crosses_acc       1571 non-null   int64  
 14  crosses_inacc     1571 non-null   int64  
 15  lballs_acc        1571 non-null   int64  
 16  lballs_inacc      1571 non-null   int64  
 17

In [18]:
from sklearn.model_selection import train_test_split
# Separar X y y
X = df_porteros.drop(columns=["original_rating"])
y = df_porteros["original_rating"]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [15]:
! pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 8.6 MB/s eta 0:00:00


In [19]:
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np
from catboost import CatBoostRegressor

# Entrenar modelo
catboost = CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6, random_seed=42, verbose=0)
catboost.fit(X_train, y_train)

# Predecir en test
y_pred = catboost.predict(X_test)

# Calcular métricas
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"CatBoost R²: {r2:.4f}")
print(f"CatBoost RMSE: {rmse:.4f}")

CatBoost R²: 0.7733
CatBoost RMSE: 0.3524


In [20]:
import pandas as pd

# Obtener importancias
importancias = catboost.get_feature_importance(prettified=True)

# Si quieres sólo las más importantes, ordena y selecciona las top N
top_features = importancias.sort_values('Importances', ascending=False)

print(top_features)  # Muestra todas ordenadas

# Opcional: mostrar sólo las 10 primeras
print(top_features.head(20))

          Feature Id  Importances
0       goals_ag_itb    27.798254
1          saves_itb    23.417921
2          saves_otb    14.386574
3       goals_ag_otb    10.713541
4               lost     4.228603
5          aerials_w     4.046907
6          saved_pen     2.817959
7            touches     1.932682
8         lballs_acc     1.678980
9         passes_acc     1.202543
10      dangmistakes     0.981482
11     interceptions     0.902093
12         poss_lost     0.866872
13               win     0.811915
14      lballs_inacc     0.755692
15      passes_inacc     0.565306
16         grduels_l     0.509498
17       countattack     0.451608
18         wasfouled     0.323980
19      is_home_team     0.302702
20        clearances     0.254525
21             fouls     0.168512
22         aerials_l     0.158336
23         grduels_w     0.139514
24           tackles     0.097982
25          owngoals     0.093771
26     game_duration     0.080924
27     dribbled_past     0.071814
28      tballs

In [21]:
import numpy as np

# Asumiendo que tienes win (1/0) y lost (1/0)
# Empate donde ni win ni lost son 1
df_porteros['result'] = np.where(df_porteros['win'] == 1, 'victory',
                 np.where(df_porteros['lost'] == 1, 'defeat', 'draw'))

# Luego elimina las columnas originales
df_porteros = df_porteros.drop(columns=['win', 'lost'])

In [22]:
columnas_seleccionadas = [

    'result', 'poss_lost', 'passes_acc', 'minutesPlayed', 'ycards', 'rcards','original_rating',
    'goals_ag_itb', 'goals_ag_otb', 'saves_itb', 'saves_otb', 'saved_pen', 'aerials_w', 'lballs_acc', 'dangmistakes'
]

df_porteros = df_porteros[columnas_seleccionadas].copy()

In [23]:
df_porteros.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1571 entries, 47 to 50613
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   result           1571 non-null   object 
 1   poss_lost        1571 non-null   int64  
 2   passes_acc       1571 non-null   int64  
 3   minutesPlayed    1571 non-null   int64  
 4   ycards           1571 non-null   int64  
 5   rcards           1571 non-null   int64  
 6   original_rating  1571 non-null   float64
 7   goals_ag_itb     1571 non-null   int64  
 8   goals_ag_otb     1571 non-null   int64  
 9   saves_itb        1571 non-null   int64  
 10  saves_otb        1571 non-null   int64  
 11  saved_pen        1571 non-null   int64  
 12  aerials_w        1571 non-null   int64  
 13  lballs_acc       1571 non-null   int64  
 14  dangmistakes     1571 non-null   int64  
dtypes: float64(1), int64(13), object(1)
memory usage: 196.4+ KB


In [28]:
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

# Separar X e y
X = df_porteros.drop(columns=['original_rating'])
y = df_porteros['original_rating']

# Dividir en train y test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Definir variables categóricas si las hay (ajustar según tu dataset)
cat_features = ['result']  # ejemplo

# Obtener índices de variables categóricas
cat_features_idx = [X_train.columns.get_loc(col) for col in cat_features if col in X_train.columns]

# Inicializar y entrenar modelo CatBoost
model = CatBoostRegressor(iterations=500, learning_rate=0.03, depth=4, random_seed=42, verbose=0,l2_leaf_reg=3)
model.fit(X_train, y_train, cat_features=cat_features_idx)

# Predecir y evaluar
y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"CatBoost R²: {r2:.4f}")
print(f"CatBoost RMSE: {rmse:.4f}")

CatBoost R²: 0.7747
CatBoost RMSE: 0.3513


In [27]:
from catboost import CatBoostRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

# Modelo base CatBoost
catboost = CatBoostRegressor(random_seed=42, verbose=0)

# Grilla de hiperparámetros pequeña para rapidez
param_grid = {
    'iterations': [200, 500],
    'depth': [4, 6],
    'learning_rate': [0.03, 0.05],
    'l2_leaf_reg': [1, 3]
}

# Configurar GridSearch
grid_search = GridSearchCV(
    estimator=catboost,
    param_grid=param_grid,
    scoring='r2',
    cv=3,
    n_jobs=-1,
    refit=True
)

# Ajustar GridSearch (pasar cat_features en fit via fit_params)
grid_search.fit(X_train, y_train, cat_features=cat_features_idx)

# Imprimir mejores parámetros y score de validación
print("Mejores parámetros:", grid_search.best_params_)
print("Mejor R² en CV:", grid_search.best_score_)

# Predecir en test con mejor modelo
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"CatBoost fine tuned R² en test: {r2:.4f}")
print(f"CatBoost fine tuned RMSE en test: {rmse:.4f}")


Mejores parámetros: {'depth': 4, 'iterations': 500, 'l2_leaf_reg': 3, 'learning_rate': 0.03}
Mejor R² en CV: 0.7786978842412147
CatBoost fine tuned R² en test: 0.7747
CatBoost fine tuned RMSE en test: 0.3513


In [29]:
df_porteros.head()

,result,poss_lost,passes_acc,minutesPlayed,ycards,rcards,original_rating,goals_ag_itb,goals_ag_otb,saves_itb,saves_otb,saved_pen,aerials_w,lballs_acc,dangmistakes
47,defeat,6,12,90,0,0,6.10,1,1,2,3,0,0,6,0
68,victory,2,11,90,0,0,6.20,1,0,1,0,0,0,5,0
192,draw,7,14,90,0,0,6.92,0,1,4,1,0,2,10,0
208,draw,6,17,90,0,0,5.99,1,0,2,0,0,0,8,0
229,victory,6,13,90,0,0,6.33,1,0,1,1,0,1,11,0


In [30]:
df_porteros.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1571 entries, 47 to 50613
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   result           1571 non-null   object 
 1   poss_lost        1571 non-null   int64  
 2   passes_acc       1571 non-null   int64  
 3   minutesPlayed    1571 non-null   int64  
 4   ycards           1571 non-null   int64  
 5   rcards           1571 non-null   int64  
 6   original_rating  1571 non-null   float64
 7   goals_ag_itb     1571 non-null   int64  
 8   goals_ag_otb     1571 non-null   int64  
 9   saves_itb        1571 non-null   int64  
 10  saves_otb        1571 non-null   int64  
 11  saved_pen        1571 non-null   int64  
 12  aerials_w        1571 non-null   int64  
 13  lballs_acc       1571 non-null   int64  
 14  dangmistakes     1571 non-null   int64  
dtypes: float64(1), int64(13), object(1)
memory usage: 196.4+ KB


In [49]:
import pandas as pd

# Caso sintético de un jugador
caso_sintetico = pd.DataFrame([{
    "result": "draw",             # 1 = ganó, 0 = perdió
    "poss_lost": 13,          # perdió 8 posesiones
    "passes_acc": 5,        # 35 pases acertados
    "minutesPlayed": 90,     # jugó todo el partido
    "ycards": 1,             # sin amarillas
    "rcards": 0,             # sin rojas
    "goals_ag_itb": 1,       # no encajó goles "inside the box"
    "goals_ag_otb": 1,       # no encajó goles "outside the box"
    "saves_itb":2,          # no es portero
    "saves_otb": 1,          # no es portero
    "saved_pen": 0,          # no paró penaltis
    "aerials_w": 1,          # ganó 3 duelos aéreos
    "lballs_acc": 11,         # 5 balones largos completados
    "dangmistakes": 0        # ningún error grave
}])





In [32]:
def predecir_limitado(model, X, min_val=0, max_val=10):
    y_pred = model.predict(X)
    return np.clip(y_pred, min_val, max_val)

In [50]:
predecir_limitado(model, caso_sintetico)

array([6.26266808])

In [51]:
model.save_model("catboost_porteros.cbm")
